# 146 — Automatización de escritorio y RPA agéntica

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("workflow", seed=146)
assert result["kind"] == "workflow"
assert result["evidence"]
show(result)


## Solución 1 — La cuenta con 1200 facturas

- No estándar: `1200·0.25 = 300` facturas.
- (a) RPA pura: 300 excepciones × 5 min = 1500 min = **25 h/mes**.
- (b) Híbrido: el extractor resuelve `300·0.92 = 276`; excepciones 24 × 5 min
  = **2 h/mes** (más el muestreo que se decida).
- (c) Errores del extractor: `300·0.08 = 24` fallos; las validaciones atrapan
  el 70 % → **~7 errores/mes entran al ERP**. Ese número es el argumento para
  el muestreo humano y el plan de reversa: el híbrido ahorra 23 h/mes pero
  introduce un tipo de error que la RPA pura no tenía (error silencioso con
  formato válido).


In [ ]:
total, conocidas, acierto, min_exc = 1200, 0.75, 0.92, 5
no_std = total * (1 - conocidas)
print("RPA pura:", no_std * min_exc / 60, "h/mes")
exc = no_std * (1 - acierto)
print("Hibrido:", exc * min_exc / 60, "h/mes")
print("errores que entran:", exc * (1 - 0.70), "por mes")


## Solución 2 — Fragilidades del selector

1. **`title='Facturas - v2.3'`**: el título contiene la versión; la
   actualización lo cambió a v2.4. Robusto: `title` con comodín
   (`'Facturas*'`) o anclar por clase de ventana.
2. **`idx=2`**: identifica el botón por posición ordinal; si añadieron un
   botón antes, el índice apunta a otro control (¡riesgo de clic
   equivocado, no solo de fallo!). Robusto: atributo estable (automation id)
   o nombre accesible.
3. **`name='btnGuardar'`**: el rediseño pudo renombrar el control o
   traducirlo (`'btnSave'`). Robusto: múltiples atributos alternativos, o un
   nodo agéntico de respaldo que localice "el botón de guardar" por
   semántica/visión y registre que el selector primario murió (auto-reparación
   con aviso, nunca silenciosa).


## Solución 3 — Determinista o agéntico

- (a) **Determinista**: entrada estructurada, regla fija, sistema crítico.
- (b) **Agéntico**: variabilidad irreducible de formatos; con validación dura
  (fecha plausible, puesto ∈ catálogo) y umbral de confianza.
- (c) **Determinista**: la matriz oficial ES la regla; delegarla a un modelo
  convierte una política de seguridad en una sugerencia.
- (d) **Agéntico** (bajo riesgo): texto libre, error tolerable, con revisión
  opcional.
- (e) **Determinista**: el log de auditoría jamás depende de inferencia.

Patrón: agente donde la entrada es variable y el error es detectable o
tolerable; determinismo donde hay reglas oficiales, sistemas críticos o
auditoría.


## Solución 4 — Workflow auditable

(a) `evidence` (hechos del episodio) y la semilla (reproducibilidad) son el
embrión de la auditoría. Faltaría para negocio real: captura del estado
**antes/después** de cada efecto, **confianza** por decisión del nodo
agéntico, identificador del ítem de negocio, y marca temporal — sin eso no se
puede reclamar, medir deriva ni revertir.
(b) El validador está en la celda: es deliberadamente duro (falla ruidosamente
en vez de continuar con un resultado incompleto), como toda validación de un
flujo de negocio.


In [ ]:
from ai_evolution.labs import run_lab

result = run_lab("workflow", seed=146)

def validacion_dura(r):
    assert r.get("kind") == "workflow", "kind incorrecto"
    assert r.get("evidence"), "sin evidencia: rechazado"
    assert r.get("limitations"), "sin limitaciones declaradas: rechazado"
    return True

print("validado:", validacion_dura(result))
